# C1 sub-objective 3 (v2): context strategies, per-turn predictions

**Question:** how do context-management strategies affect intent accuracy against token cost and latency on
Sinhala-English code-switched, multi-turn citizen dialogue? This notebook trains the models and saves **one
prediction per test turn**; the analysis (JGA through the state tracker, code-mixing buckets, statistics,
charts) runs locally in `research/experiments/context_eval.py`.

| Strategy | What the model reads |
|---|---|
| `current_only` | only the current message |
| `window_2` / `window_3` | the last 2 / 3 user messages |
| `keyword_summary` | English words and names (EN/NE tags) from earlier messages + the current message |
| `structured_state` | the previous intent and slots so far (e.g. `intent: nic_update \| district: Matara`) + the current message |
| `summary` | **new:** earlier sentences with a service or slot cue in Sinhala, Singlish **or** English (≤ 48 XLM-R tokens) + the current message |
| `full_history` | every user message so far |

LLM-based summary: future work (not run, to avoid API cost).

The input builders are in `experiments/context_strategies.py`, shared with the local scripts. `keyword_summary`
and `structured_state` read the gold language/entity tags (an oracle, as in Observation 8); `summary` reads only the text.

**Data (v2, 12 intents).** The 977-turn silver set is split **by dialogue** (`data/synthetic/context_v2_split.json`,
seed 0, stratified by intent): 80% for training, 20% (79 dialogues, 199 turns) held out as the **synthetic test set**.
The dev set (best epoch) is 10% of the 80% part, chosen per seed; the test part is never used for training or
model selection. Dialogues are joined into sessions of 2–4 **within each part only**, so a test session never
contains a training dialogue. Test sessions are fixed (seed 0) for every strategy and seed.

| Test set | Turns | Note |
|---|---|---|
| `gold` | 142 | development set for the tracker rules: report separately |
| `synthetic` | 199 | held out from intent training and from the rules |
| `fresh` | 50 | human-written, held out |

**Models:** XLM-R (`xlm-roberta-base`, 15 epochs) and TF-IDF + logistic regression, seeds 1, 2, 3.

## Before running
1. **Mount Drive first:** run the first code cell and allow access. All inputs and outputs live in `MyDrive/c1_context_v2/`.
2. **GPU:** Runtime → Change runtime type → **T4 GPU**.
3. **Inputs:** put these 9 files in `MyDrive/c1_context_v2/inputs/` (or upload them when asked; they are then copied there):
   - from `research/data/`: `gold_tokens.jsonl`, `fresh_tokens.jsonl`, `seed_all.json`, `fresh_dialogues_states.json`
   - from `research/data/synthetic/`: `silver_tokens.jsonl`, `synthetic_dialogues.json`, `context_v2_split.json`
   - from `research/experiments/`: `context_strategies.py`, `c1_dst.py`
4. Runtime → **Run all**. Expected runtime: **about 1.5–2 hours** on a T4 (21 seed × strategy runs, each trains
   XLM-R for 15 epochs and TF-IDF, then predicts 391 test turns one at a time). If Colab disconnects, Run all again:
   finished runs are skipped.

## Outputs (`MyDrive/c1_context_v2/`)
- `preds/<model>__<strategy>__seed<seed>.jsonl`: **42 files** (2 models × 7 strategies × 3 seeds), 391 lines each
  (gold 142, synthetic 199, fresh 50). One line per test turn: test set, session, dialogue, turn index, text,
  bot text, model input, gold and predicted intent, confidence, input tokens (XLM-R and tiktoken `cl100k_base`),
  context tokens, latency.
- `runs.json`: training history and quick scores per run.
- Download the `preds/` folder to `research/results/context_v2/preds/` for `context_eval.py`.

W&B is optional and off by default: set `C1_WANDB=1` and add `WANDB_API_KEY` as a Colab secret (🔑 in the left bar).

In [ ]:
# 1. Google Drive (run first) and the input files
import os, sys, json, shutil

try:
    from google.colab import drive
    drive.mount("/content/drive")
    ON_COLAB = True
except ImportError:                       # local run: set C1_OUT to a folder that has inputs/ in it
    ON_COLAB = False
OUT_DIR = os.environ.get("C1_OUT", "/content/drive/MyDrive/c1_context_v2")
IN_DIR, PRED_DIR = os.path.join(OUT_DIR, "inputs"), os.path.join(OUT_DIR, "preds")
for d in (IN_DIR, PRED_DIR):
    os.makedirs(d, exist_ok=True)

INPUTS = ["silver_tokens.jsonl", "gold_tokens.jsonl", "fresh_tokens.jsonl", "seed_all.json",
          "synthetic_dialogues.json", "fresh_dialogues_states.json", "context_v2_split.json",
          "context_strategies.py", "c1_dst.py"]
missing = [f for f in INPUTS if not os.path.exists(os.path.join(IN_DIR, f))]
if missing and ON_COLAB:
    print("Please upload:", missing)
    from google.colab import files
    for name, content in files.upload().items():
        open(os.path.join(IN_DIR, os.path.basename(name)), "wb").write(content)
    missing = [f for f in INPUTS if not os.path.exists(os.path.join(IN_DIR, f))]
if missing:
    raise FileNotFoundError(f"missing in {IN_DIR}: {missing}")

sys.path.insert(0, IN_DIR)
import importlib, context_strategies as cs
importlib.reload(cs)
print("inputs:", IN_DIR, "| predictions go to:", PRED_DIR)

In [ ]:
# 2. Settings
import random, time, collections, statistics
import numpy as np
import torch
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup

MODEL = os.environ.get("C1_MODEL", "xlm-roberta-base")
SEEDS = [int(s) for s in os.environ.get("C1_SEEDS", "1,2,3").split(",")]
EPOCHS = int(os.environ.get("C1_EPOCHS", 15))
STRATEGIES = os.environ.get("C1_STRATEGIES", ",".join(cs.STRATEGIES)).split(",")
BATCH, MAX_LEN, LR, DEV_SHARE = 16, 128, 3e-5, 0.1
STATE_NOISE = 0.2         # structured_state training: share of turns where the previous intent is randomised
SMOKE = os.environ.get("C1_SMOKE") == "1"            # plumbing test only: 30 training dialogues
USE_WANDB = os.environ.get("C1_WANDB", "0") == "1"   # off by default
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE, "| model:", MODEL, "| seeds:", SEEDS, "| epochs:", EPOCHS, "| strategies:", STRATEGIES)
if DEVICE == "cpu" and not SMOKE:
    print("WARNING: no GPU. Runtime -> Change runtime type -> T4 GPU, then rerun.")

try:
    import tiktoken
except ImportError:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tiktoken"], check=True)
    import tiktoken
cl100k = tiktoken.get_encoding("cl100k_base")

wb = None
if USE_WANDB:
    try:
        from google.colab import userdata
        key = userdata.get("WANDB_API_KEY")            # Colab secret; never typed into the notebook
        import wandb
        wandb.login(key=key)
        wb = wandb.init(project="helaguide-c1", name="context-v2",
                        config={"model": MODEL, "epochs": EPOCHS, "seeds": SEEDS, "strategies": STRATEGIES})
    except Exception as e:
        print("W&B not used:", e)
        wb = None


def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

In [ ]:
# 3. Data: synthetic split, test sessions, bot text before each user turn


def read_jsonl(name):
    with open(os.path.join(IN_DIR, name), encoding="utf-8") as fh:
        return [json.loads(l) for l in fh if l.strip()]


def read_json(name):
    with open(os.path.join(IN_DIR, name), encoding="utf-8") as fh:
        return json.load(fh)


silver, gold, fresh = read_jsonl("silver_tokens.jsonl"), read_jsonl("gold_tokens.jsonl"), read_jsonl("fresh_tokens.jsonl")
split = read_json("context_v2_split.json")
TRAIN_IDS, TEST_IDS = set(split["train"]), set(split["test"])
assert not TRAIN_IDS & TEST_IDS, "a dialogue is in both parts of the split"
assert {r["dialogue_id"] for r in silver} == TRAIN_IDS | TEST_IDS, "split does not match silver_tokens.jsonl"
syn_train = [r for r in silver if r["dialogue_id"] in TRAIN_IDS]
syn_test = [r for r in silver if r["dialogue_id"] in TEST_IDS]

INTENTS = sorted({r["intent"] for r in syn_train})
t2i = {l: i for i, l in enumerate(INTENTS)}; i2t = {i: l for l, i in t2i.items()}
assert len(INTENTS) == 12, INTENTS

BOT = {}                  # (dialogue_id, turn index) -> the bot message just before that user turn (None at the start)
for name in ("seed_all.json", "synthetic_dialogues.json", "fresh_dialogues_states.json"):
    for d in read_json(name):
        bot = None
        for i, t in enumerate(d["turns"]):
            if t.get("speaker") == "bot":
                bot = t["text"]
            else:
                BOT[(d["dialogue_id"], i)] = bot

# test sessions are fixed (seed 0), built inside each test set only
TEST_SETS = {"gold": cs.stitched_sessions(gold, 0), "synthetic": cs.stitched_sessions(syn_test, 0),
             "fresh": cs.stitched_sessions(fresh, 0)}
for name, sessions in TEST_SETS.items():
    rows = [r for s in sessions for r in s]
    no_bot = [r for r in rows if (r["dialogue_id"], r["turn"]) not in BOT]
    assert not no_bot, f"{name}: {len(no_bot)} turns not found in the dialogue files"
    lens = [len(s) for s in sessions]
    print(f"{name:10s} {len(rows):4d} turns | {len(sessions):3d} sessions | turns per session mean {np.mean(lens):.1f}, max {max(lens)}"
          f" | intents not in training: {sorted({r['intent'] for r in rows} - set(INTENTS))}")
print(f"synthetic: train part {len(syn_train)} turns ({len(TRAIN_IDS)} dialogues), test {len(syn_test)} turns ({len(TEST_IDS)} dialogues)")

In [ ]:
# 4. Tokens, inputs and one example session through every strategy
tok = AutoTokenizer.from_pretrained(MODEL)
tok.truncation_side = "left"     # if the input is too long, drop the OLDEST text, never the current message


def n_xlmr(text):                # tokens the model reads (with <s> and </s>), before truncation
    return len(tok(text, truncation=False)["input_ids"])


def n_plain(text):               # used for the 48-token summary cap
    return len(tok(text, add_special_tokens=False)["input_ids"])


def n_cl100k(text):
    return len(cl100k.encode(text))


def build(strategy, prev, cur, prev_intent):
    return cs.build_input(strategy, prev, cur, prev_intent, count_tokens=n_plain)


demo = TEST_SETS["gold"][0]
for st in STRATEGIES:
    x = build(st, demo[:-1], demo[-1], demo[-2]["intent"] if len(demo) > 1 else None)
    print(f"\n[{st}] turn {len(demo)}, {n_xlmr(x)} tokens:\n  {x}")

In [ ]:
# 5. Training and prediction helpers


def training_examples(sessions, strategy, rng):
    xs, ys = [], []
    for s in sessions:
        for pos, cur in enumerate(s):
            prev_intent = s[pos - 1]["intent"] if pos else None
            if strategy == "structured_state" and prev_intent and rng.random() < STATE_NOISE:
                prev_intent = rng.choice(INTENTS)     # teaches the model not to blindly copy the previous intent
            xs.append(build(strategy, s[:pos], cur, prev_intent)); ys.append(t2i[cur["intent"]])
    return xs, ys


def macro_f1(golds, preds):
    f1s = []
    for l in INTENTS:
        tp = sum(g == l and p == l for g, p in zip(golds, preds))
        fp = sum(g != l and p == l for g, p in zip(golds, preds))
        fn = sum(g == l and p != l for g, p in zip(golds, preds))
        pr = tp / (tp + fp) if tp + fp else 0.0
        rc = tp / (tp + fn) if tp + fn else 0.0
        f1s.append(2 * pr * rc / (pr + rc) if pr + rc else 0.0)
    return sum(f1s) / len(f1s)


@torch.no_grad()
def xlmr_batch(model, inputs, bs=32):
    model.eval(); preds = []
    for i in range(0, len(inputs), bs):
        enc = tok(inputs[i:i + bs], truncation=True, max_length=MAX_LEN, padding=True, return_tensors="pt").to(model.device)
        preds += [i2t[x] for x in model(**enc).logits.argmax(-1).cpu().tolist()]
    return preds


def dev_score(model, sessions, strategy):
    """Dev macro-F1, turn by turn; structured_state reads the model's own previous prediction."""
    golds, preds, prev = [], [], [None] * len(sessions)
    for pos in range(max(len(s) for s in sessions)):
        live = [(i, s) for i, s in enumerate(sessions) if len(s) > pos]
        out = xlmr_batch(model, [build(strategy, s[:pos], s[pos], prev[i]) for i, s in live])
        for (i, s), p in zip(live, out):
            prev[i] = p; golds.append(s[pos]["intent"]); preds.append(p)
    return macro_f1(golds, preds)


def collate(batch):
    n = max(len(b["input_ids"]) for b in batch)
    pad = tok.pad_token_id
    return {"input_ids": torch.tensor([b["input_ids"] + [pad] * (n - len(b["input_ids"])) for b in batch]),
            "attention_mask": torch.tensor([b["attention_mask"] + [0] * (n - len(b["attention_mask"])) for b in batch]),
            "labels": torch.tensor([b["labels"] for b in batch])}


def train_xlmr(xs, ys, dev_fn, seed):
    set_seed(seed)
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL, num_labels=len(INTENTS), id2label=i2t, label2id=t2i).float().to(DEVICE)
    enc = [tok(x, truncation=True, max_length=MAX_LEN) for x in xs]
    data = [{"input_ids": e["input_ids"], "attention_mask": e["attention_mask"], "labels": y} for e, y in zip(enc, ys)]
    dl = DataLoader(data, batch_size=BATCH, shuffle=True, collate_fn=collate)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    steps = len(dl) * EPOCHS
    sched = get_linear_schedule_with_warmup(opt, int(0.1 * steps), steps)
    best, best_state, hist = -1, None, []
    for ep in range(1, EPOCHS + 1):
        model.train(); t0, total = time.time(), 0.0
        for b in dl:
            b = {k: v.to(DEVICE) for k, v in b.items()}
            loss = model(**b).loss
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); sched.step(); opt.zero_grad()
            total += loss.item()
        score = dev_fn(model)
        hist.append({"epoch": ep, "loss": round(total / len(dl), 4), "dev": round(score, 4)})
        print(f"    epoch {ep}: loss {total / len(dl):.4f}  dev macro-F1 {score:.4f}  ({time.time() - t0:.0f}s)")
        if score > best:
            best, best_state = score, {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    return model, hist


def train_tfidf(xs, ys):
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.linear_model import LogisticRegression
    from sklearn.pipeline import make_pipeline, make_union
    m = make_pipeline(
        make_union(TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), sublinear_tf=True),
                   TfidfVectorizer(analyzer="word", ngram_range=(1, 2), sublinear_tf=True)),
        LogisticRegression(max_iter=3000, C=10))
    m.fit(xs, [i2t[y] for y in ys])
    return m


@torch.no_grad()
def xlmr_one(model, text):
    """Batch size 1: (intent, confidence, latency in ms: tokenise + forward)."""
    t0 = time.perf_counter()
    enc = tok(text, truncation=True, max_length=MAX_LEN, return_tensors="pt").to(model.device)
    probs = model(**enc).logits.softmax(-1)[0]
    j = int(probs.argmax())
    conf = float(probs[j])                    # .item() waits for the GPU, so the time below is complete
    return i2t[j], conf, (time.perf_counter() - t0) * 1000


def tfidf_one(m, text):
    t0 = time.perf_counter()
    p = m.predict_proba([text])[0]
    j = int(p.argmax())
    return m.classes_[j], float(p[j]), (time.perf_counter() - t0) * 1000


def predict_turns(one, model_name, strategy, seed):
    """Every test turn, session by session, one at a time (structured_state reads its own previous prediction)."""
    one(TEST_SETS["gold"][0][0]["text"])      # warm-up
    out = []
    for ts, sessions in TEST_SETS.items():
        for si, s in enumerate(sessions):
            prev_pred = None
            for pos, cur in enumerate(s):
                t0 = time.perf_counter()
                x = build(strategy, s[:pos], cur, prev_pred)
                build_ms = (time.perf_counter() - t0) * 1000
                pred, conf, ms = one(x)
                prev_pred = pred
                nx, nc = n_xlmr(x), n_cl100k(x)
                out.append({
                    "model": model_name, "strategy": strategy, "seed": seed, "test_set": ts,
                    "session_id": f"{ts}-{si:03d}", "session_pos": pos, "dialogue_id": cur["dialogue_id"],
                    "turn_index": cur["turn"], "code_mix": cur.get("code_mix"), "text": cur["text"],
                    "bot_text": BOT[(cur["dialogue_id"], cur["turn"])], "input": x,
                    "gold_intent": cur["intent"], "pred_intent": pred, "confidence": round(conf, 4),
                    "tokens_xlmr": nx, "tokens_cl100k": nc,
                    "context_tokens_xlmr": nx - n_xlmr(cur["text"]), "context_tokens_cl100k": nc - n_cl100k(cur["text"]),
                    "truncated": nx > MAX_LEN, "latency_ms": round(ms, 3), "build_ms": round(build_ms, 3),
                    "device": DEVICE if model_name == "xlmr" else "cpu"})
    return out


def save_preds(rows, path):
    tmp = path + ".tmp"                       # written in full, then renamed: a half-written file never counts as done
    with open(tmp, "w", encoding="utf-8") as fh:
        for r in rows:
            fh.write(json.dumps(r, ensure_ascii=False) + "\n")
    os.replace(tmp, path)


def quick(rows):
    return {ts: {"n": len(v), "intent_acc": round(sum(r["gold_intent"] == r["pred_intent"] for r in v) / len(v), 4),
                 "tokens_xlmr_mean": round(float(np.mean([r["tokens_xlmr"] for r in v])), 1),
                 "latency_ms_mean": round(float(np.mean([r["latency_ms"] for r in v])), 2)}
            for ts in TEST_SETS for v in [[r for r in rows if r["test_set"] == ts]]}

## Run the experiment
For each seed and strategy: train XLM-R and TF-IDF on the training sessions, then predict every test turn.
Each model's predictions are saved as soon as they are ready; a model whose file exists is skipped on a rerun.

In [ ]:
RUNS_PATH = os.path.join(OUT_DIR, "runs.json")
runs = json.load(open(RUNS_PATH, encoding="utf-8")) if os.path.exists(RUNS_PATH) else {}


def pred_path(model_name, strategy, seed):
    return os.path.join(PRED_DIR, f"{model_name}__{strategy}__seed{seed}.jsonl")


def save_runs():
    json.dump(runs, open(RUNS_PATH, "w", encoding="utf-8"), ensure_ascii=False, indent=2)


for seed in SEEDS:
    # dev = 10% of the TRAINING part of the split, chosen per seed; the synthetic test part is never touched
    ids = sorted(TRAIN_IDS)
    random.Random(seed).shuffle(ids)
    if SMOKE:
        ids = ids[:40]
    dev_ids = set(ids[: max(1, int(len(ids) * DEV_SHARE))])
    fit_ids = set(ids) - dev_ids
    assert not (dev_ids | fit_ids) & TEST_IDS
    train_s = cs.stitched_sessions([r for r in syn_train if r["dialogue_id"] in fit_ids], seed)
    dev_s = cs.stitched_sessions([r for r in syn_train if r["dialogue_id"] in dev_ids], seed)

    for st in STRATEGIES:
        key = f"seed{seed}/{st}"
        todo = [m for m in ("xlmr", "tfidf") if not os.path.exists(pred_path(m, st, seed))]
        if not todo:
            print(f"{key}: already done, skipping"); continue
        print(f"\n=== {key} ({', '.join(todo)}) ===")
        xs, ys = training_examples(train_s, st, random.Random(seed))
        run = runs.get(key, {"train_examples": len(xs), "train_dialogues": len(fit_ids), "dev_dialogues": len(dev_ids)})

        if "tfidf" in todo:
            tfidf = train_tfidf(xs, ys)
            rows = predict_turns(lambda x: tfidf_one(tfidf, x), "tfidf", st, seed)
            save_preds(rows, pred_path("tfidf", st, seed))
            run["tfidf"] = quick(rows)
            print("  TF-IDF:", run["tfidf"])

        if "xlmr" in todo:
            t0 = time.time()
            model, hist = train_xlmr(xs, ys, lambda m: dev_score(m, dev_s, st), seed)
            rows = predict_turns(lambda x: xlmr_one(model, x), "xlmr", st, seed)
            save_preds(rows, pred_path("xlmr", st, seed))
            run["xlmr"] = {**quick(rows), "train_minutes": round((time.time() - t0) / 60, 1), "history": hist}
            print("  XLM-R:", {k: v for k, v in run["xlmr"].items() if k != "history"})
            del model
            if DEVICE == "cuda":
                torch.cuda.empty_cache()

        runs[key] = run
        save_runs()
        if wb:
            wb.log({f"{m}/{ts}/{st}/intent_acc": run[m][ts]["intent_acc"] for m in ("xlmr", "tfidf") if m in run
                    for ts in TEST_SETS} | {"seed": seed})
print("\nAll runs finished.")

## Check: files and a quick look (intent accuracy only; JGA and statistics are in `context_eval.py`)

In [ ]:
expected = [pred_path(m, st, s) for s in SEEDS for st in STRATEGIES for m in ("xlmr", "tfidf")]
found = [p for p in expected if os.path.exists(p)]
print(f"{len(found)} of {len(expected)} prediction files in {PRED_DIR}")
for p in sorted(set(expected) - set(found)):
    print("  missing:", os.path.basename(p))

table = collections.defaultdict(list)
for p in found:
    rows = [json.loads(l) for l in open(p, encoding="utf-8")]
    for ts, v in quick(rows).items():
        table[(rows[0]["model"], rows[0]["strategy"], ts)].append((v["intent_acc"], v["tokens_xlmr_mean"]))
fmt = lambda xs: f"{statistics.mean(xs):.3f}±{statistics.stdev(xs) if len(xs) > 1 else 0:.3f}"
for ts in TEST_SETS:
    print(f"\n{ts}: intent accuracy, mean±std over seeds")
    print(f"  {'strategy':18s}{'tokens':>8}{'XLM-R':>15}{'TF-IDF':>15}")
    for st in STRATEGIES:
        x, t = table.get(("xlmr", st, ts), []), table.get(("tfidf", st, ts), [])
        tokens = (x or t or [(0, 0)])[0][1]
        print(f"  {st:18s}{tokens:>8}{fmt([a for a, _ in x]) if x else '-':>15}{fmt([a for a, _ in t]) if t else '-':>15}")
if wb:
    wb.finish()